# Active Learning Experiments

This notebook uses precomputed embeddings along with the active learning simulation utilities in the repository. You can vary classifiers, sampling strategies, diversity settings, seed sizes, and plot learning curves to compare behaviors.

## 1. Imports and helper functions

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from src.active_learning.simulate_embeddings import run_al_simulation
sns.set_theme(style='whitegrid')

## 2. Run a simulation
Specify parameters and run `run_al_simulation`. You can choose a sampling strategy (`entropy`, `margin`, `least_confidence`,`random`) and optionally enable diversity. Results will be saved in the embeddings parent folder unless you provide `out_dir`.

In [ ]:
# example parameters
emb_dir = Path('artifacts/embedding_cnn/embeddings')
results = run_al_simulation(embeddings_dir=emb_dir, seed_size=10, rounds=10, query_size=10, strategy='entropy', diversity=False, seed=0)
# `results` is a dict of classifier->metrics etc.
results.keys()

## 3. Load and plot learning curves
After running simulations you can read the metrics CSVs and plot test accuracy over rounds.

In [ ]:
import glob
all_metrics = []
for csv in glob.glob(str(emb_dir.parent / 'al_embeddings_results' / '*_metrics.csv')):
    df = pd.read_csv(csv)
    clf = Path(csv).stem.replace('_metrics','')
    df['classifier'] = clf
    all_metrics.append(df)
if all_metrics:
    metrics_df = pd.concat(all_metrics, ignore_index=True)
    display(metrics_df.head())
    plt.figure(figsize=(8,4))
    sns.lineplot(data=metrics_df, x='round', y='test_acc', hue='classifier', marker='o')
    plt.title('Learning curves (accuracy)')
    plt.show()
else:
    print('No metrics files found. Run simulation cell first.')

## 4. Experiment with different settings
Modify the parameters below and rerun the simulation cell to compare strategies and models.

In [ ]:
# try several strategies with and without diversity
emb_dir = Path('../../artifacts/embedding_cnn/embeddings')
for strat in ['entropy','least_confidence','margin']:
    print('strategy', strat)
    run_al_simulation(embeddings_dir=emb_dir, seed_size=80, rounds=5, query_size=50, strategy=strat, diversity=False, seed=1)
    run_al_simulation(embeddings_dir=emb_dir, seed_size=80, rounds=5, query_size=50, strategy=strat, diversity=True, seed=1)
    print('---')

## 5. Additional ideas
- Change `seed_size` or `query_size` to see impact of initial labelled set and batch size.
- Add/remove classifiers inside `run_al_simulation` (edit source or copy function).
- Compute and plot F1 score similarly.
- Use the queries CSVs to inspect which examples were selected each round.